In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType

from mastodon_utils import get_spark, KAFKA_BOOTSTRAP, TOPIC, CHECKPOINT_DIR, JDBC_URL, JDBC_PROPS

spark = get_spark("mastodon-streaming")

Spark 3.5.3 prêt - Spark UI : http://localhost:4040


In [2]:
# le schema d un toot(fichier json envoyé par le producteur ) 
toot_schema = StructType([
    StructField("toot_id", StringType()),
    StructField("created_at", StringType()),
    StructField("user_id", StringType()),
    StructField("username", StringType()),
    StructField("acct", StringType()),
    StructField("instance", StringType()),
    StructField("followers_count", LongType()),
    StructField("text", StringType()),
    StructField("language", StringType()),
    StructField("hashtags", ArrayType(StringType())),
    StructField("favourites_count", LongType()),
    StructField("reblogs_count", LongType()),
    StructField("replies_count", LongType()),
    StructField("url", StringType()),
    StructField("location", StringType()),
    StructField("source", StringType()),
    StructField("ingested_at", StringType()),
])

In [3]:
raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
toots = (
    raw.select(
        F.from_json(F.col("value").cast("string"), toot_schema).alias("t"),
        "partition", "offset",
    )
    .select("t.*", "partition", "offset")
    .filter(F.col("toot_id").isNotNull()) 
    .withColumn("event_time", F.to_timestamp("created_at")) 
    .withColumn("text", F.trim("text"))
    .withColumn("text_length", F.length("text"))
)

In [5]:
LANGUAGES = ["en", "fr"]
KEYWORDS_REGEX = r"(?i)\b(data|ai|spark|kafka|python|machine learning|big data|llm)\b"

filtered = (
    toots
    .filter(F.col("language").isin(LANGUAGES))
    .filter(F.col("text_length") > 0)
    .withColumn("has_keyword", F.col("text").rlike(KEYWORDS_REGEX))
)

In [6]:
toots_per_minute = (
    filtered
    .withWatermark("event_time", "10 minutes")   # on accepte jusqu'à 10 min de retard
    .groupBy(F.window("event_time", "1 minute"), "language")
    .count()
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "language",
        F.col("count").alias("nb_toots"),
    )
)

In [7]:
stats_by_hashtag = (
    filtered
    .select(F.explode("hashtags").alias("hashtag"), "text_length")#1 ligne par hashtag du toot
    .groupBy("hashtag")
    .agg(
        F.count("*").alias("nb_toots"),
        F.round(F.avg("text_length"), 1).alias("avg_length"),
    )
)

stats_by_user = (
    filtered
    .groupBy("user_id", "username")
    .agg(
        F.count("*").alias("nb_toots"),
        F.round(F.avg("text_length"), 1).alias("avg_length"),
    )
)

In [8]:
def append_to(table):
    def _write(batch_df, batch_id):
        batch_df.write.mode("append").jdbc(JDBC_URL, table, properties=JDBC_PROPS)
    return _write


def overwrite(table):
    def _write(batch_df, batch_id):
        (batch_df.write.mode("overwrite").option("truncate", "true")
            .jdbc(JDBC_URL, table, properties=JDBC_PROPS))
    return _write

In [9]:

toots_out = filtered.select(
    "toot_id", "event_time", "user_id", "username", "instance", "language", "text",
    "text_length", F.array_join("hashtags", ",").alias("hashtags"),
    "favourites_count", "reblogs_count", "replies_count", "followers_count",
    "has_keyword", "url", "partition", "offset",
)

q_toots = (
    toots_out.writeStream.queryName("toots")
    .outputMode("append")
    .foreachBatch(append_to("toots"))
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/toots")
    .trigger(processingTime="10 seconds")
    .start()
)

q_minute = (
    toots_per_minute.writeStream.queryName("toots_per_minute")
    .outputMode("append")
    .foreachBatch(append_to("toots_per_minute"))
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/toots_per_minute")
    .trigger(processingTime="10 seconds")
    .start()
)

q_hashtag = (
    stats_by_hashtag.writeStream.queryName("stats_by_hashtag")
    .outputMode("complete")
    .foreachBatch(overwrite("stats_by_hashtag"))
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/stats_by_hashtag")
    .trigger(processingTime="30 seconds")
    .start()
)

q_user = (
    stats_by_user.writeStream.queryName("stats_by_user")
    .outputMode("complete")
    .foreachBatch(overwrite("stats_by_user"))
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/stats_by_user")
    .trigger(processingTime="30 seconds")
    .start()
)

In [10]:
for q in spark.streams.active:
    p = q.lastProgress
    print(f"{q.name:18} actif={q.isActive}  lignes au dernier batch={p['numInputRows'] if p else '-'}")

toots              actif=True  lignes au dernier batch=1159
stats_by_user      actif=True  lignes au dernier batch=1159
stats_by_hashtag   actif=True  lignes au dernier batch=1159
toots_per_minute   actif=True  lignes au dernier batch=0
